In [3]:
"""
Apply step 2's entity resolution decisions (approved_lookup_table) to
final_resolved_triples.xlsx, producing the fully resolved entity
columns. This is the general-synonym counterpart to
apply_entity_merges.py, which applied the sample-code (tier2/tier3)
merges, this applies the broader cross-corpus synonym merges (soy vs
soybean, plural/singular, etc.) from step1/step2.

Pipeline this closes out: step1_generate_entity_candidates.py ->
step2_validate_entity_groups.py -> (this script).

Only applies rows that are BOTH is_synonym_group = True AND
needs_manual_review = False, matching step2's own approved_lookup_table
sheet exactly, nothing here re-decides what step2 already decided.

Designed for Jupyter/Colab execution. No __main__ guard.
"""

import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "final_resolved_triples.xlsx"
SOURCE_COL = "final_source"
TARGET_COL = "final_target"

STEP2_OUTPUT_PATH = "entity_resolution_review.xlsx"
STEP2_CANDIDATE_SHEET = "candidate_groups"  # reads directly from here, not approved_lookup_table

OUTPUT_TRIPLES_XLSX = "fully_resolved_triples.xlsx"
OUTPUT_AUDIT_XLSX = "entity_resolution_audit.xlsx"

# ---------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------
df = pd.read_excel(TRIPLES_PATH)
print(f"Loaded {len(df)} triples")

# Reads candidate_groups directly, not the auto-generated
# approved_lookup_table. This means ANY row you've manually reviewed
# and confirmed gets applied, even if the LLM originally disagreed,
# just flip is_synonym_group to TRUE and needs_manual_review to FALSE
# for that row in the spreadsheet, no separate override list needed.
candidates_df = pd.read_excel(STEP2_OUTPUT_PATH, sheet_name=STEP2_CANDIDATE_SHEET)
approved = candidates_df[
    (candidates_df["is_synonym_group"] == True) &  # noqa: E712
    (candidates_df["needs_manual_review"] == False)  # noqa: E712
]
print(f"Groups approved (is_synonym_group=True, needs_manual_review=False): {len(approved)} "
      f"of {len(candidates_df)} total group rows in candidate_groups")

lookup = {}
lookup_rows_for_audit = []
for _, row in approved.iterrows():
    canonical = row["proposed_canonical_name"]
    for m in str(row["members"]).split("; "):
        m = m.strip()
        lookup[m] = canonical
        lookup_rows_for_audit.append({"raw_name": m, "canonical_name": canonical,
                                        "group_id": row["group_id"]})

lookup_df = pd.DataFrame(lookup_rows_for_audit)
print(f"Loaded {len(lookup)} approved raw_name -> canonical_name mappings")

# ---------------------------------------------------------------
# APPLY
# ---------------------------------------------------------------
def resolve(entity):
    key = str(entity).strip()
    return lookup.get(key, entity), (key in lookup)


resolved_sources, resolved_targets = [], []
source_changed, target_changed = [], []

for _, row in df.iterrows():
    rs, rs_changed = resolve(row[SOURCE_COL])
    rt, rt_changed = resolve(row[TARGET_COL])
    resolved_sources.append(rs)
    resolved_targets.append(rt)
    source_changed.append(rs_changed)
    target_changed.append(rt_changed)

df["resolved_source"] = resolved_sources
df["resolved_target"] = resolved_targets
df["source_entity_resolved"] = source_changed
df["target_entity_resolved"] = target_changed

n_source = sum(source_changed)
n_target = sum(target_changed)
print(f"\nRows with source resolved: {n_source}")
print(f"Rows with target resolved: {n_target}")

# ---------------------------------------------------------------
# NODE COUNT BEFORE/AFTER
# ---------------------------------------------------------------
before_nodes = pd.concat([df[SOURCE_COL], df[TARGET_COL]]).astype(str).str.strip().nunique()
after_nodes = pd.concat([df["resolved_source"], df["resolved_target"]]).astype(str).str.strip().nunique()
print(f"\nUnique nodes before this pass: {before_nodes}")
print(f"Unique nodes after this pass: {after_nodes}")
print(f"Nodes collapsed: {before_nodes - after_nodes}")

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
df.to_excel(OUTPUT_TRIPLES_XLSX, index=False)
print(f"\nSaved fully resolved triples to {OUTPUT_TRIPLES_XLSX}")

readme_rows = [
    "HOW TO READ THIS FILE",
    "",
    "- merge_lookup: every (raw_name -> canonical_name) pair actually applied, straight from",
    "  step2's approved_lookup_table, no re-decision made here.",
    "",
    f"Nodes before this pass: {before_nodes}. Nodes after: {after_nodes}. Collapsed: {before_nodes - after_nodes}.",
    "",
    "resolved_source/resolved_target in fully_resolved_triples.xlsx are the final entity",
    "columns, use these (not final_source/final_target) for any downstream graph analysis",
    "from here on.",
]
readme_df = pd.DataFrame({"": readme_rows})

with pd.ExcelWriter(OUTPUT_AUDIT_XLSX) as writer:
    readme_df.to_excel(writer, sheet_name="READ_ME_FIRST", index=False)
    lookup_df.to_excel(writer, sheet_name="merge_lookup", index=False)

print(f"Saved audit workbook to {OUTPUT_AUDIT_XLSX}")

Loaded 10324 triples
Groups approved (is_synonym_group=True, needs_manual_review=False): 217 of 217 total group rows in candidate_groups
Loaded 555 approved raw_name -> canonical_name mappings

Rows with source resolved: 2477
Rows with target resolved: 2361

Unique nodes before this pass: 3100
Unique nodes after this pass: 2759
Nodes collapsed: 341

Saved fully resolved triples to fully_resolved_triples.xlsx
Saved audit workbook to entity_resolution_audit.xlsx
